In [1]:
import pandas as pd
import numpy as np
import os


pi = 3.14159265359

maxval=1e9
minval=1e-9

In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from models.mlp_encoder_model_nonquantized import *

2025-10-03 15:38:19.740909: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-03 15:38:20.516621: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
model=CreateModel_Slim((16,16,2))
model.summary()

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['input_pxls[0][0]']          
 Pooling2D)                                                                                       
                                                                                                  
 average_pooling2d_1 (Avera  (None, 1, 16, 2)             0         ['input_pxls[0][0]']          
 gePooling2D)                                                                                     
                                                                                 

2025-10-03 15:38:24.447071: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1209 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB MIG 1g.5gb, pci bus id: 0000:c1:00.0, compute capability: 8.0


In [5]:
# get best weights file
pitch = '50x12P5'
batch_size = 5000
fingerprint = '34c2da80'
timeslices = 2
files = os.listdir('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))

vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
bestfile = files[np.argmin(vlosses)]
model.load_weights('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)

print('Best model: {}'.format(bestfile))

Best model: weights.2000-t33.87-v32.40.hdf5


In [6]:
# load in the test set
test_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    quantize = False # False for soft quantizer and manually quantized inputs
)

Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM/metadata.json


In [7]:
# predicts test data
p_test = model.predict(test_generator)

complete_truth = None
for _, y in test_generator:
    if complete_truth is None:
        complete_truth = y
    else:
        complete_truth = np.concatenate((complete_truth, y), axis=0)

# creates df with all predicted values and matrix elements - 4 predictions, all 10 unique matrix elements
df = pd.DataFrame(p_test,columns=['x','y','cotB'])

# stores all true values in same matrix as xtrue, ytrue, etc.
df['xtrue'] = complete_truth[:,0]
df['ytrue'] = complete_truth[:,1]
df['cotBtrue'] = complete_truth[:,2]

# calculates residuals for x, y, cotA, cotB
df['residualsX'] = df['xtrue'] - df['x']
df['residualsY'] = df['ytrue'] - df['y']
df['residualsB'] = df['cotBtrue'] - df['cotB']

# stores results as parquet
df.to_parquet("/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))

 1/21 [>.............................] - ETA: 5s

2025-10-03 15:39:55.707637: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8906
2025-10-03 15:39:55.713060: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


21/21 [==============================] - 6s 270ms/step
